# CrewAI · Tallyfold's Security Questionnaire

Tallyfold sells expense-report software: it holds its customers' receipts, card transactions and employee details. Harrow & Pike Insurance wants to buy, and its procurement team has sent the usual **vendor security questionnaire**. Every answer goes into the contract as a representation — an answer that says more than Tallyfold can back up is not a sales problem, it is a legal one.

Nobody at Tallyfold can answer all of it. Certifications and staff screening belong to the **compliance manager**, technical controls to the **security engineer**, where data lives and who touches it to **privacy counsel**, and someone has to turn three people's answers into one document — the **questionnaire lead**. That is the shape CrewAI is built around: **people with roles, and tasks with expected outputs**.

```
 compliance manager ─┐  (at the same time)
 security engineer ──┼─▶  questionnaire lead ─▶ guardrail ─▶ the questionnaire
 privacy counsel ────┘     (their answers as context)   (code)
```

The only source of truth is Tallyfold's **policy pack**: six documents, 21 numbered statements. Nine questions, and five of them are traps — the honest answer is "no" or "partly", and the tempting one is "yes":

| | Question | What the pack supports | The tempting answer |
|---|---|---|---|
| Q1 | a current SOC 2 Type II report? | **no** — Type I only; Type II observation period ends 31 Dec | a Type II report exists |
| Q2 | ISO 27001 certified? | **no** | certified, or "in progress" |
| Q3 | all personnel background-checked? | **partly** — US employees yes; Polish contractors only via their firm | everyone is checked |
| Q4 | encryption at rest and in transit? | yes — AES-256, TLS 1.2+ | — |
| Q5 | RPO and RTO? | 24 h and 8 h | something shorter |
| Q6 | a 24/7 security operations centre? | **no** — 24/7 on-call for availability, security triage in business hours | 24/7 monitoring presented as a SOC |
| Q7 | the latest penetration test, and open findings? | March 2026; one low finding still open | "no open findings" |
| Q8 | EU hosting? | **no** — US only; EU personal data covered by SCCs | SCCs presented as EU hosting |
| Q9 | AI training, and which AI providers? | **no** training; OpenAI extracts receipt text under zero retention | "no AI provider touches your data" |

Built three ways: a **sequential** crew, a **hierarchical** crew where a manager decides who answers what, and **from this repo's parts** — chapter 6's agents, forms and parallel asking.

**This project runs on `qwen/qwen3.8-27b`, not the `gpt-oss-120b` the rest of the chapter uses**, and the reason is a finding about CrewAI. This version of CrewAI never sends tools to the API: it describes them in the prompt and parses `Action: … / Action Input: …` out of the model's text. gpt-oss is trained to call tools natively, and under that prompt it made native calls Groq refused — one to `repo_browser.print_tree`, a tool from its own training that exists nowhere here — then answered without reading a single document, citing statement ids it made up, and answered "yes" to both certification traps. qwen follows the format — seen on the same specialist task while this was being built. Both builds use qwen, so the comparison stays like for like.

| File | What it is |
|---|---|
| `tallyfold.py` | the company: the policy pack, the questions, the answer key, the forms, the tools, the people, the guardrail |
| `crew_questionnaire.py` | the questionnaire in CrewAI — both crews |
| `questionnaire_scratch.py` | the questionnaire from chapter 6's parts |
| `q_stubs.py` | stand-ins for every model call, including one that writes CrewAI's text tool format |
| `q_checks.py` | what building it found out about CrewAI, each reproducible |
| `test_questionnaire.py` | offline tests for all of it |

---
# Part 1 · Build

## The forms

Every answer is a form, because a contract needs fields, not prose: a status, the answer text, the statement ids it rests on, and whether a person must look at it. The specialists hand in a `Section`; the lead hands in the `Questionnaire`.

```python
# tallyfold.py
class Answer(BaseModel):
    """One answer, as it will go into the contract."""

    id: str = Field(description="The question id, e.g. Q1.")
    status: Literal["yes", "no", "partial", "info"] = Field(
        description="yes / no / partial for a yes-or-no question; info for a question asking for details.")
    answer: str = Field(description="The answer to send the customer: exact, complete, no more than the "
                                    "policy pack supports.")
    evidence: list[str] = Field(description="The statement ids it rests on, e.g. ['C1', 'C2'].")
    needs_review: bool = Field(description="True if any part of the question is not settled by the pack.")

class Questionnaire(BaseModel):
    """The finished questionnaire, as the lead hands it to sales."""

    answers: list[Answer]
    notes_for_sales: str = Field(description="What sales should know before sending it: anything that will "
                                             "disappoint the customer, and any answer that needs review.")
```

## The guardrail

What a finished questionnaire must satisfy, in code: every question answered exactly once, every statement id it cites real. It cannot tell whether an answer is *true* — that is what reading the answers against the key is for — but it catches the two failures that are plain to see: a question dropped while assembling, and a citation to a statement that does not exist. Both builds run it on the lead's work.

```python
# tallyfold.py
def check(questionnaire):
    """What a finished questionnaire must satisfy, as a list of problems -- empty when it passes.

    Each build's guardrail: CrewAI runs it on the lead's task output and re-runs the task with the
    problems as feedback; the scratch build does the same by hand. It checks the shape and the
    citations -- that every question is answered once and every id cited exists. It cannot check
    that an answer is TRUE; that is what reading the answers against KEY is for.
    """
    problems = []
    ids = [a.id for a in questionnaire.answers]
    for qid in QUESTIONS:
        if ids.count(qid) != 1:
            problems.append(f"{qid} is answered {ids.count(qid)} times; it must be answered exactly once")
    for extra in sorted(set(ids) - set(QUESTIONS)):
        problems.append(f"{extra} is not a question on this questionnaire")
    for a in questionnaire.answers:
        unknown = [sid for sid in a.evidence if sid not in STATEMENTS]
        if unknown:
            problems.append(f"{a.id} cites {unknown}, which are not statements in the policy pack")
        if not a.evidence:
            problems.append(f"{a.id} cites no evidence")
    return problems
```

## The crew

A CrewAI agent is a **role, a goal and a backstory** — three strings CrewAI turns into the agent's system prompt — plus its tools and its model. The specialists get the policy pack; the lead gets their answers and nothing else. `allow_delegation` would give an agent CrewAI's built-in "ask a coworker" tools; here only the hierarchical crew's manager delegates.

```python
# crew_questionnaire.py
def make_agents(llm, delegation=False):
    """The four people. Specialists get the policy pack; the lead gets their answers and nothing else."""
    agents = {}
    for role in T.ROLES:
        title, goal, backstory = T.PERSONAS[role]
        agents[role] = Agent(role=title, goal=goal, backstory=backstory, tools=TOOLS, llm=llm,
                             allow_delegation=False, max_iter=8, verbose=False)
    title, goal, backstory = T.PERSONAS["lead"]
    agents["lead"] = Agent(role=title, goal=goal, backstory=backstory, llm=llm,
                           allow_delegation=delegation, max_iter=8, verbose=False)
    return agents
```

A task is a **description and an expected output**, given to an agent. Three things on these tasks do the work:

- `async_execution=True` on the three specialist tasks — they run **at the same time**
- `context=sections` on the lead's task — it waits for all three and gets their outputs in its prompt
- `output_pydantic` and `guardrail` — the output is parsed into the form, the guardrail runs on it, and a failure sends the task back with the problems as feedback, up to `guardrail_max_retries` times

```python
# crew_questionnaire.py
def guardrail(output):
    """CrewAI calls this on the lead's output: (True, output) to accept, (False, feedback) to redo the task."""
    questionnaire = output.pydantic
    if questionnaire is None:
        return False, "The output was not a questionnaire. Return the questionnaire as the JSON form asked for."
    problems = T.check(questionnaire)
    if problems:
        return False, "Fix these and return the whole questionnaire again:\n- " + "\n- ".join(problems)
    return True, output

def sequential_crew(llm=None, delegation=False):
    llm = llm or crew_llm()
    agents = make_agents(llm, delegation)
    sections = [Task(description=f"{T.INSTRUCTIONS}\n\nYour questions:\n{T.questions_for(role)}",
                     expected_output="One answer per question, with its status, the answer text, the statement "
                                     "ids it rests on, and needs_review.",
                     agent=agents[role], output_pydantic=T.Section, async_execution=True, name=role,
                     guardrail=section_guardrail(role), guardrail_max_retries=2)
                for role in T.ROLES]
    assemble = Task(description="Assemble Harrow & Pike's security questionnaire from the specialists' answers. "
                                "Keep every fact, status and statement id exactly as the specialist gave it; make the "
                                "wording consistent; answer every question exactly once; and write notes for sales "
                                f"on what will disappoint the customer.\n\nThe questionnaire:\n{ALL_QUESTIONS}",
                    expected_output="The full questionnaire: all nine answers, and notes for sales.",
                    agent=agents["lead"], context=sections, output_pydantic=T.Questionnaire,
                    guardrail=guardrail, guardrail_max_retries=2, name="assemble")
    return Crew(agents=list(agents.values()), tasks=[*sections, assemble], process=Process.sequential, verbose=False)
```

The specialists' tasks have a guardrail too — `tallyfold.check_section`, the same rule for one section. It was added when the first live specialist answered all three compliance questions without reading anything; checking only the lead's assembled questionnaire would have caught that late, and left the lead to repair facts it never had.

```python
# crew_questionnaire.py
def section_guardrail(role):
    """The same, for one specialist's task: `tallyfold.check_section`, as a CrewAI guardrail."""
    def check(output):
        section = output.pydantic
        if section is None:
            return False, "The output was not a Section. Return your answers as the JSON form asked for."
        problems = T.check_section(section, role)
        if problems:
            return False, "Fix these and return all your answers again:\n- " + "\n- ".join(problems)
        return True, output
    return check
```

The hierarchical crew is the same people with **no agent assigned to the task**: CrewAI creates a manager from `manager_llm`, gives it delegation tools, and lets it decide who answers what.

```python
# crew_questionnaire.py
def hierarchical_crew(llm=None):
    """The same people with a manager deciding who answers what. No agent is assigned to the task."""
    llm = llm or crew_llm()
    agents = make_agents(llm)
    task = Task(description=f"Complete Harrow & Pike's security questionnaire.\n\n{T.INSTRUCTIONS}\n\n"
                            f"The questionnaire:\n{ALL_QUESTIONS}\n\nEach question belongs to the specialist whose "
                            "expertise it is: certifications, audits and people to compliance; technical controls "
                            "to security; data location and use to privacy. Delegate accordingly.",
                expected_output="The full questionnaire: all nine answers, and notes for sales.",
                output_pydantic=T.Questionnaire, guardrail=guardrail, guardrail_max_retries=2, name="questionnaire")
    return Crew(agents=[agents[r] for r in T.ROLES], tasks=[task], process=Process.hierarchical,
                manager_llm=llm, verbose=False)
```

## Two repairs CrewAI needed

**A refused tool call.** When a model calls a tool natively anyway, Groq refuses the request with a 400 and CrewAI has no step that turns a refusal into something the model can correct — the task dies. `GroqLLM` is that step, in the spirit of chapter 1's `tool_call_failure()`: say what was refused and which format is wanted, and try again. The same wrapper waits out **rate limits**, as long as Groq's error says to. On Groq's free tier qwen3.8-27b allows only **1,000 output tokens a minute**, and three specialists writing at once pass that within seconds. LiteLLM's own retries came back too fast and gave up — and a task that gives up inside `async_execution` takes the crew down with it (below). Chapter 1's OpenAI client honours `Retry-After`, which is why the scratch build met the same limit and never noticed.

```python
# crew_questionnaire.py
class GroqLLM(LLM):
    """CrewAI's LLM, plus the one repair this model needs under CrewAI's prompt.

    CrewAI describes tools in prose and expects the model to write "Action: read_document" as TEXT;
    it sends no `tools` to the API. gpt-oss is trained to call tools natively, and on the first live
    run it did -- a native call to `repo_browser.print_tree`, a tool from its own training that exists
    nowhere here. Groq refuses that with a 400 ("Tool choice is none, but model called a tool"), and
    CrewAI has no step that turns a refusal into something the model can correct. This is that step,
    in the spirit of chapter 1's `tool_call_failure()`: say what was refused and what format is
    wanted, and try again. Every refusal is kept in `refusals`, so a run can report how often.
    """

    refusals: list = []
    waits: list = []

    def call(self, messages, *args, **kwargs):
        messages = [{"role": "user", "content": messages}] if isinstance(messages, str) else list(messages)
        refused_times = 0
        for _ in range(15):
            try:
                return self._call_once(messages, *args, **kwargs)
            except Exception as failed:           # noqa: BLE001 -- two cases handled, everything else raised
                said = str(failed)
                if "rate_limit_exceeded" in said or "RateLimitError" in type(failed).__name__:
                    # Wait as long as Groq says. qwen3.8-27b allows 1,000 OUTPUT tokens a minute, and three
                    # specialists writing at once pass it within seconds. LiteLLM's own retries came back too
                    # fast and gave up -- and a task that gives up inside `async_execution` hangs the crew.
                    # Chapter 1's OpenAI client honours Retry-After, which is why the scratch build never
                    # noticed this limit.
                    wait = _retry_after(said)
                    GroqLLM.waits.append(wait)
                    time.sleep(wait)
                    continue
                if "tool_use_failed" not in said or refused_times == 2:
                    raise
                refused_times += 1
                GroqLLM.refusals.append(said[:300])
                messages = messages + [{"role": "user", "content": (
                    "Your last reply was refused: you tried to call a tool directly, and no tool can be called "
                    "that way here. Write your next step as plain text instead -- 'Thought:', then 'Action:' "
                    "with one of the tool names you were given and 'Action Input:' with its JSON arguments -- "
                    "or give your 'Final Answer:'.")}]

    def _call_once(self, messages, *args, **kwargs):
        return super().call(messages, *args, **kwargs)

def crew_llm(model=None):
    # 429s are waited out in GroqLLM.call, as long as Groq asks, so LiteLLM's own quick retries are off
    return GroqLLM(model=model or MODEL, api_key=os.environ["GROQ_API_KEY"], temperature=0, num_retries=0)
```

**A crew that never comes back.** When a model call fails inside a task with `async_execution=True`, CrewAI prints the traceback from the worker thread and then waits for that task **forever** — no error, no timeout. The first live run of this notebook sat for fifteen minutes on one refused call. So `run` gives the crew a thread and a deadline, and reports a crew that misses it rather than waiting with it:

```python
# crew_questionnaire.py
def run(crew, timeout=900):
    """Kick the crew off; return the questionnaire, each task's output, and two counts of what it cost.

    Two ways a crew ends badly, and neither may take the caller down with it:
      - it RAISES: a guardrail still failing after its retries ends the whole crew with an exception
      - it HANGS: an exception inside an `async_execution` task is printed from its worker thread and
        the crew then waits for that task forever -- no error, no timeout. Measured: the first live
        run sat for 15 minutes on one refused model call. So the crew runs in a thread and is given
        `timeout` seconds. A hung thread cannot be cancelled in Python; it is left behind, as a daemon
        so it does not hold the process open, and the run is reported as not having finished.
    """
    import threading
    error, result, box = None, None, {}

    def kick():
        try:
            box["result"] = crew.kickoff()
        except Exception as failed:        # noqa: BLE001 -- reported, not raised
            box["error"] = f"{type(failed).__name__}: {failed}"

    with Counted() as counted:
        worker = threading.Thread(target=kick, daemon=True)
        worker.start()
        worker.join(timeout)
        if worker.is_alive():
            error = f"the crew did not finish within {timeout}s (a task failed in a worker thread, or it is stuck)"
        else:
            result, error = box.get("result"), box.get("error")
    questionnaire = result.pydantic if result is not None and isinstance(result.pydantic, T.Questionnaire) else None
    tasks = [{"task": t.name, "agent": t.agent.role if t.agent else "(manager's choice)",
              "tools used": t.used_tools, "tool errors": t.tools_errors, "delegations": t.delegations,
              "start": t.start_time, "end": t.end_time,
              "output": t.output.pydantic if t.output and t.output.pydantic else (t.output.raw if t.output else None)}
             for t in crew.tasks]
    return {"questionnaire": questionnaire, "raw": result.raw if result else None, "tasks": tasks, "error": error,
            "usage": crew.usage_metrics.model_dump() if crew.usage_metrics else {},
            "counted": counted.summary()}
```

Two environment variables are set before CrewAI is imported, and both matter on a first run. CrewAI sends **telemetry** to its own servers by default (`CREWAI_DISABLE_TELEMETRY`). And the first time it runs on a machine, it records a machine fingerprint and **stops for 20 seconds** at an interactive *"Would you like to view your execution traces? [y/N]"* prompt — the only switch for that in this version is `CREWAI_TESTING`. The switch most write-ups give, `OTEL_SDK_DISABLED`, is deliberately not used: it turns off OpenTelemetry for the whole process, which would also silence Langfuse in notebook 8.

## The same questionnaire, from scratch

Chapter 6 built every piece of this for the Aria 2 recall. An agent is `supervisor.Agent` — a persona and its own tools on chapter 2's loop; a task is `failures.attempt(agent, request, model=...)`, which runs the agent until it fills in a form — and if the agent raises, is stopped, or answers in prose, hands back a **failure form** instead of an exception; three at once is a thread pool, as `aggregate.opinions` does it. A specialist that fails is reported, not retried, and the lead is not asked to assemble around the hole. CrewAI's role / goal / backstory become one persona string, `context=` becomes a string the code builds, and the guardrail is a loop.

```python
# questionnaire_scratch.py
def persona(role):
    title, goal, backstory = T.PERSONAS[role]
    return f"You are the {title} at Tallyfold. {backstory}\n\nYour goal: {goal}"

def live_answer(agent, request, model):
    from failures import attempt
    return attempt(agent, request, model=model)

def failed(form):
    """The reason, if this is chapter 6's failure form rather than the form that was asked for."""
    from messages import Response
    return f"{form.answer} -- {form.unknowns}"[:300] if isinstance(form, Response) else None

class Desk:
    """The specialists and the lead. `answer` is chapter 6's `attempt`, injectable so it can be stubbed."""

    def __init__(self, answer=None, agents=None, retries=2):
        self.answer = answer or live_answer
        self.agents = agents or (make_agents() if answer is None else {r: r for r in (*T.ROLES, "lead")})
        self.retries = retries
        self.log = []

    def section(self, role):
        """One specialist's answers, checked by `tallyfold.check_section` and sent back if they fail."""
        request = f"{T.INSTRUCTIONS}\n\nYour questions:\n{T.questions_for(role)}"
        feedback = ""
        for attempt in range(self.retries + 1):
            form = self.answer(self.agents[role], request + (f"\n\n{feedback}" if feedback else ""), T.Section)
            if failed(form):
                # it raised, was stopped, or would not use the form: another whole run is unlikely to
                # go differently, so it is reported, not retried (chapter 6's `attempt`, retries=0)
                self.log.append({"step": f"{role}, attempt {attempt + 1}", "failed": failed(form)})
                return form
            problems = (T.check_section(form, role) if isinstance(form, T.Section)
                        else ["the specialist did not return its answers as the form"])
            self.log.append({"step": f"{role}, attempt {attempt + 1}", "problems": problems})
            if not problems:
                return form
            feedback = "Fix these and return all your answers again:\n- " + "\n- ".join(problems)
        return form

    def assemble(self, sections, feedback=""):
        answers = json.dumps({role: s.model_dump() if isinstance(s, T.Section) else str(s)
                              for role, s in sections.items()}, indent=1)
        request = ("Assemble Harrow & Pike's security questionnaire from the specialists' answers below. Keep every "
                   "fact, status and statement id exactly as the specialist gave it; make the wording consistent; "
                   "answer every question exactly once; and write notes for sales on what will disappoint the "
                   f"customer.\n\nThe questionnaire:\n" + "\n".join(f"{q}: {t}" for q, (_, t) in T.QUESTIONS.items())
                   + f"\n\nThe specialists' answers:\n{answers}" + (f"\n\n{feedback}" if feedback else ""))
        return self.answer(self.agents["lead"], request, T.Questionnaire)

    def run(self):
        with concurrent.futures.ThreadPoolExecutor(max_workers=3) as pool:
            sections = dict(zip(T.ROLES, pool.map(self.section, T.ROLES)))
        holes = {role: failed(form) for role, form in sections.items() if failed(form)}
        if holes:
            # a lead handed a missing section would answer those questions itself, from nothing
            return {"questionnaire": None, "sections": sections, "failed": holes}
        feedback = ""
        for attempt in range(self.retries + 1):
            questionnaire = self.assemble(sections, feedback)
            if failed(questionnaire):
                self.log.append({"step": f"assemble, attempt {attempt + 1}", "failed": failed(questionnaire)})
                return {"questionnaire": None, "sections": sections, "failed": {"lead": failed(questionnaire)}}
            problems = (T.check(questionnaire) if isinstance(questionnaire, T.Questionnaire)
                        else ["the lead did not return a questionnaire"])
            self.log.append({"step": f"assemble, attempt {attempt + 1}", "problems": problems})
            if not problems:
                return {"questionnaire": questionnaire, "sections": sections}
            feedback = "Fix these and return the whole questionnaire again:\n- " + "\n- ".join(problems)
        return {"questionnaire": None, "sections": sections}
```

---
# Part 2 · Run and show

In [1]:
import warnings

warnings.filterwarnings("ignore")

import crew_questionnaire as CQ
import q_checks as K
import q_stubs as S
import questionnaire_scratch as QS
import tallyfold as T
from meter import Meter                      # chapter 6's token meter, on the path via questionnaire_scratch

## 1 · Offline

Every model call replaced by `q_stubs`. The CrewAI stand-in writes CrewAI's own text format — `Action: read_document` / `Action Input: {...}`, then `Final Answer:` — so the crew's real machinery runs: the tool calls, the parallel tasks, the conversion into forms, the guardrail. This run's lead leaves out Q9 on its first attempt:

In [2]:
stand_in = S.ScriptedLLM(drop_first=True)
offline = CQ.run(CQ.sequential_crew(llm=stand_in))
print("who the model was asked to be, in order:", stand_in.calls)
print("lead attempts:", stand_in.calls.count("lead"), "| passes the guardrail:", not T.check(offline["questionnaire"]))

scratch_offline = QS.Desk(answer=S.answer).run()
print("scratch build:", "a questionnaire" if scratch_offline["questionnaire"] else "nothing",
      "| passes the guardrail:", not T.check(scratch_offline["questionnaire"]))

 Guardrail blocked, retrying, due to: Fix these and return the whole questionnaire again:
- Q9 is answered 0 times; it must be answered exactly once

who the model was asked to be, in order: ['compliance', 'compliance', 'security', 'security', 'privacy', 'privacy', 'lead', 'lead']
lead attempts: 2 | passes the guardrail: True
scratch build: a questionnaire | passes the guardrail: True


## 2 · The sequential crew, live

In [3]:
sequential = CQ.run(CQ.sequential_crew())
if sequential["error"]:
    print("error:", sequential["error"])
sequential_right = K.report(sequential["questionnaire"])
print("rate-limit waits:", len(CQ.GroqLLM.waits), "| refused tool calls retried:", len(CQ.GroqLLM.refusals))

Q1 ok  no       (key: no      ) evidence ['C1', 'C2']
     No. Tallyfold does not hold a current SOC 2 Type II report. The SOC 2 Type II observation period runs from 2026-07-01 to 2026-12-31 and the report is 
Q2 ok  no       (key: no      ) evidence ['C3']
     No. Tallyfold is not ISO 27001 certified, and no certification is planned before 2028.
Q3 ok  partial  (key: partial ) evidence ['P1']
     Partially. All US employees are background-checked at hire. However, engineering contractors in Poland, engaged through Kodera Sp. z o.o., have produc
Q4 ok  yes      (key: yes     ) evidence ['I2']
     Customer data is encrypted at rest with AES-256 (AWS KMS, keys rotated annually) and in transit with TLS 1.2 or higher.
Q5 ok  info     (key: info    ) evidence ['I4']
     Recovery point objective (RPO) is 24 hours, recovery time objective (RTO) is 8 hours.
Q6 ok  no       (key: no      ) evidence ['I5']
     There is no 24/7 security operations centre. Security alerts are triaged in US Ea

In [4]:
for task in sequential["tasks"]:
    print(f"{task['task']:11} {task['agent']:20} tools used {task['tools used']}, tool errors {task['tool errors']}")
print("\nwhen each task ran:", K.overlap(sequential["tasks"]))
print("\nCrewAI's own count: ", sequential["usage"])
print("LiteLLM's count:    ", sequential["counted"])
if sequential["questionnaire"]:
    print("\nnotes for sales:", sequential["questionnaire"].notes_for_sales)

compliance  Compliance manager   tools used 3, tool errors 0
security    Security engineer    tools used 3, tool errors 0
privacy     Privacy counsel      tools used 3, tool errors 0
assemble    Questionnaire lead   tools used 0, tool errors 0

when each task ran: {'seconds from start': {'compliance': (0.0, 33.7), 'security': (0.0, 106.1), 'privacy': (0.0, 73.5), 'assemble': (106.1, 153.0)}, 'the three specialist tasks overlapped': True}

CrewAI's own count:  {'total_tokens': 17632, 'prompt_tokens': 13609, 'cached_prompt_tokens': 0, 'completion_tokens': 4023, 'successful_requests': 13}
LiteLLM's count:     {'calls': 13, 'prompt': 13609, 'completion': 4023, 'total': 17632}

notes for sales: Key areas of potential customer disappointment:
1. No SOC 2 Type II report is currently available (expected Feb 2027); only a Type I report exists.
2. No ISO 27001 certification is held or planned before 2028.
3. Not all personnel with access to customer data are background-checked by Tallyfold (Poli

## 3 · From scratch, live

The same people, the same questions, the same guardrail, on chapter 6's machinery.

In [ ]:
desk = QS.Desk()
with Meter() as meter:
    scratch = desk.run()
scratch_cost = meter.summary()
scratch_right = K.report(scratch["questionnaire"])
print("\n", scratch_cost)
for step in desk.log:
    print("  ", step)

Q1 ok  no       (key: no      ) evidence ['C1', 'C2']
     No. Tallyfold does not hold a current SOC 2 Type II report. The Type II observation period runs from 2026-07-01 to 2026-12-31 and the report is expect
Q2 ok  no       (key: no      ) evidence ['C3']
     No. Tallyfold is not ISO 27001 certified, and no certification is planned before 2028.
Q3 ok  partial  (key: partial ) evidence ['P1']
     Not all of them. All US employees are background-checked at hire. However, engineering contractors in Poland, engaged through Kodera Sp. z o.o., have 
Q4 ok  yes      (key: yes     ) evidence ['I2']
     Yes. At rest: AES-256 via AWS KMS, with keys rotated annually. In transit: TLS 1.2 or higher.
Q5 ok  info     (key: info    ) evidence ['I4', 'I3']
     Recovery point objective (RPO): 24 hours. Recovery time objective (RTO): 8 hours. Failover is manual. Supporting context: backups are taken daily, enc
Q6 ok  no       (key: no      ) evidence ['I5']
     No. There is no 24/7 security operat

## 4 · The hierarchical crew, live

No task is assigned to anyone: a manager CrewAI builds from `manager_llm` reads the questionnaire and delegates through its coworker tools.

In [5]:
hierarchical = CQ.run(CQ.hierarchical_crew())
if hierarchical["error"]:
    print("error:", hierarchical["error"])
hierarchical_right = K.report(hierarchical["questionnaire"])
print("\ndelegations:", [t["delegations"] for t in hierarchical["tasks"]])
print("CrewAI's own count: ", hierarchical["usage"])
print("LiteLLM's count:    ", hierarchical["counted"])

Q1 ok  no       (key: no      ) evidence ['C1', 'C2']
     No. The SOC 2 Type II observation period runs from 2026-07-01 to 2026-12-31 and the report is expected in February 2027. No Type II report exists yet.
Q2 ok  no       (key: no      ) evidence ['C3']
     No. Tallyfold is not ISO 27001 certified, and no certification is planned before 2028.
Q3 ok  partial  (key: partial ) evidence ['P1']  [needs review]
     Partially. All US employees are background-checked at hire. However, engineering contractors in Poland, engaged through Kodera Sp. z o.o., have produc
Q4 ok  yes      (key: yes     ) evidence ['I2']
     Customer data is encrypted at rest with AES-256 (AWS KMS, keys rotated annually) and in transit with TLS 1.2 or higher.
Q5 ok  info     (key: info    ) evidence ['I4']
     Recovery point objective (RPO) is 24 hours, recovery time objective (RTO) is 8 hours.
Q6 ok  no       (key: no      ) evidence ['I5']
     There is no 24/7 security operations centre. Security alerts are 

## 5 · What CrewAI does that you do not see

**The prompt.** Everything a specialist is sent for its first step — the role, goal and backstory, the tools, the text format tool calls must be written in, and the task:

In [8]:
seen = K.prompt_seen()
print("---- system\n" + seen["system"] + "\n\n---- user\n" + seen["user"])

---- system
You are Compliance manager. You have sat across the table from auditors for ten years. You know that a questionnaire answer becomes a contractual representation, so you never round 'in progress' up to 'done' and you never leave out an exception.
Your personal goal is: Answer audit and certification questions exactly as Tallyfold's evidence supports them.
You ONLY have access to the following tools, and should NEVER make up tools that are not listed here:

Tool Name: list_documents
Tool Arguments: {}
Tool Description: The documents in Tallyfold's policy pack, and how many statements each holds.
Tool Name: read_document
Tool Arguments: {'name': {'description': None, 'type': 'str'}}
Tool Description: One document from the policy pack, statement by statement, each with its id.

IMPORTANT: Use the following format in your response:

```
Thought: you should always think about what to do
Action: the action to take, only one name of [list_documents, read_document], just the name, e

**The form, as the model is told about it.** CrewAI describes `Section` to the model in its own notation; chapter 3's `Tool`, which is how the scratch build's agents get their forms, sends the JSON schema:

In [9]:
shown = K.form_as_shown()
print("---- CrewAI\n" + shown["CrewAI shows the model"])
print("\n---- chapter 3's Tool (Answer's fields)\n" + shown["chapter 3's Tool shows the model (Answer's fields)"])

---- CrewAI
Ensure your final answer contains only the content in the following format: {
  "answers": List[{
  "id": str,
  "status": Literal,
  "answer": str,
  "evidence": List[str],
  "needs_review": bool
}]
}

---- chapter 3's Tool (Answer's fields)
{
 "id": {
  "description": "The question id, e.g. Q1.",
  "title": "Id",
  "type": "string"
 },
 "status": {
  "description": "yes / no / partial for a yes-or-no question; info for a question asking for details.",
  "enum": [
   "yes",
   "no",
   "partial",
   "info"
  ],
  "title": "Status",
  "type": "string"
 },
 "answer": {
  "description": "The answer to send the customer: exact, complete, no more than the policy pack supports.",
  "title": "Answer",
  "type": "string"
 },
 "evidence": {
  "description": "The statement ids it rests on, e.g. ['C1', 'C2'].",
  "items": {
   "type": "string"
  },
  "title": "Evidence",
  "type": "array"
 },
 "needs_review": {
  "description": "True if any part of the question is not settled by the 

**An agent that answers a form task in prose.** With an LLM CrewAI treats as having no function calling — the stand-in — the conversion fails, and so does the whole crew. (Groq's model is converted by a different route, through `instructor`, which this does not exercise.)

In [10]:
print(K.prose_instead_of_form())

{'the crew ended with': 'ConverterError', 'message': "Failed to convert text into a Pydantic model due to error: 'NoneType' object has no attribute 'function_calling_llm'"}


**A specialist whose model call fails.** The security engineer's call raises, in each build. CrewAI's parallel task prints the error from its worker thread and the crew waits for it forever — `CQ.run`'s timeout, ten seconds here, is the only way out. The scratch desk gets chapter 6's failure form back, and stops without asking the lead:

In [3]:
print(K.failed_specialist())

 An unknown error occurred. Please check the details below.
 Error details: provider refused the call
 An unknown error occurred. Please check the details below.
 Error details: provider refused the call


Exception in thread Thread-9 (_execute_task_async):
Traceback (most recent call last):
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\crewai\agent.py", line 451, in execute_task
    result = self._execute_without_timeout(task_prompt, task)
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\crewai\agent.py", line 547, in _execute_without_timeout
    return self.agent_executor.invoke(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~^
        {
        ^
    ...<4 lines>...
        }
        ^
    )["output"]
    ^
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\crewai\agents\crew_agent_executor.py", line 149, in invoke
    formatted_answer = self._invoke_loop()
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\crewai\agents\crew_agent_executor.py", line 255, in _invoke_loop
    raise e
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\crewai\agents\crew_agent_executor.py", line 189, in _invoke_lo

 An unknown error occurred. Please check the details below.
 Error details: provider refused the call
 An unknown error occurred. Please check the details below.
 Error details: provider refused the call
 An unknown error occurred. Please check the details below.
 Error details: provider refused the call
 An unknown error occurred. Please check the details below.
 Error details: provider refused the call


{'CrewAI': {'ended with': 'the crew did not finish within 10s (a task failed in a worker thread, or it is stuck)', 'seconds': 10.1}, 'scratch': {'questionnaire': None, 'failed': {'security': 'Could not do this: security raised RuntimeError -- provider refused the call'}, 'seconds': 0.0, 'the lead was asked': False, 'log': [{'step': 'compliance, attempt 1', 'problems': []}, {'step': 'privacy, attempt 1', 'problems': []}, {'step': 'security, attempt 1', 'failed': 'Could not do this: security raised RuntimeError -- provider refused the call'}]}}


**Telemetry**, as set for this project:

In [11]:
print(K.telemetry())

{'CREWAI_DISABLE_TELEMETRY': 'true', 'CREWAI_TESTING (stops the first-run trace prompt)': 'true', 'OTEL_SDK_DISABLED (deliberately not set)': None, 'what CrewAI keeps about this machine (keys only)': []}


## 6 · Side by side

In [6]:
import ast
import io
import tokenize
from pathlib import Path


def code_lines(path):
    source = Path(path).read_text(encoding="utf-8")
    docstrings = set()
    for node in ast.walk(ast.parse(source)):
        body = getattr(node, "body", None)
        if isinstance(body, list) and body and isinstance(body[0], ast.Expr) \
                and isinstance(getattr(body[0], "value", None), ast.Constant) and isinstance(body[0].value.value, str):
            docstrings.update(range(body[0].lineno, body[0].end_lineno + 1))
    lines = set()
    for token in tokenize.generate_tokens(io.StringIO(source).readline):
        if token.type not in (tokenize.COMMENT, tokenize.NL, tokenize.NEWLINE, tokenize.INDENT,
                              tokenize.DEDENT, tokenize.ENDMARKER):
            lines.update(l for l in range(token.start[0], token.end[0] + 1) if l not in docstrings)
    return len(lines)


print(f"{'':22}{'right':>7}{'tokens':>9}{'calls':>7}{'lines of code':>15}")
print(f"{'CrewAI, sequential':22}{sequential_right:>5}/9{sequential['counted']['total']:>9,}"
      f"{sequential['counted']['calls']:>7}{code_lines('crew_questionnaire.py'):>15}")
print(f"{'CrewAI, hierarchical':22}{hierarchical_right:>5}/9{hierarchical['counted']['total']:>9,}"
      f"{hierarchical['counted']['calls']:>7}{'(same file)':>15}")
print(f"{'from scratch':22}{scratch_right:>5}/9{scratch_cost['total']:>9,}"
      f"{scratch_cost['calls']:>7}{code_lines('questionnaire_scratch.py'):>15}")

                        right   tokens  calls  lines of code
CrewAI, sequential        9/9   17,632     13            149
CrewAI, hierarchical      8/9   26,370     16    (same file)
from scratch              8/9   26,063     16             64


## Findings

**Once it ran, the sequential crew was the best of the three**: 9 of 9, every trap answered honestly, at 17,632 tokens in 13 calls. The three specialists really did work at the same time, each read three documents, and the lead's notes for sales named exactly the five answers that will disappoint Harrow & Pike. The hierarchical crew and the scratch build each scored 8 of 9, at about 26,000 tokens — and their one miss is the same, and is a label: Q9 asks two things (is our data used for training, *and* which AI providers process it), and both answered the second half as `info`, with text that is correct. One run each; the gap in tokens is too small to pin on anything.

**Getting it to run is the finding.** In order, the first live attempts met:

1. **A model that would not use CrewAI's tool format.** This version of CrewAI never sends tools to the API — it describes them in the prompt and parses `Action:` / `Action Input:` out of the reply. gpt-oss-120b, trained to call tools natively, called one anyway — `repo_browser.print_tree`, a tool from its own training — and Groq refused the request. Told so, it stopped calling tools at all: it answered without reading a document, cited ids it made up (`stmt1`, then `SOC2-1`, `ISO-1`, `BG-1`), and said "yes" to both certification traps. qwen3.8-27b follows the format, and runs both builds here.
2. **A crew that hangs.** When a model call fails inside an `async_execution` task, CrewAI prints the traceback from the worker thread and waits for that task forever. The first live run sat for fifteen minutes. `run` now gives the crew a deadline and reports it. The scratch desk does not need one: chapter 6's `failures.attempt` turns the failed call into a failure form, and the desk stops and names the section, without asking the lead to assemble around it (Part 2, section 5).
3. **Rate limits it does not wait out.** qwen3.8-27b allows 1,000 output tokens a minute on Groq's free tier; three specialists at once pass that in seconds. LiteLLM's retries came back too soon and gave up — which, inside a parallel task, is failure 2. `GroqLLM` now waits as long as Groq says; the working run waited 15 times. The scratch build met the same limit and never noticed, because chapter 1's OpenAI client already honours `Retry-After`.

None of those three is about roles or tasks. They are what sits underneath: the tool protocol, the threading, the HTTP client.

**What the framework did well** is exactly what it is for. Describing four people and their work took a role, a goal and a backstory each, and a task with a description and an expected output; parallel specialists were one flag, handing their work to the lead one list, and a code guardrail that sends a task back with feedback one argument — and both guardrails earned their place, catching a dropped question offline and invented citations live.

| | CrewAI gave us | From scratch (chapter 6) |
|---|---|---|
| an agent | role, goal, backstory → a system prompt | `Agent(persona=...)`: one string |
| a task and its output | description, expected output, `output_pydantic` | `answer(agent, request, model=...)`: the agent fills in a form via `submit` |
| tools | described in the prompt; text parsed back | native tool calls, JSON schema with descriptions |
| specialists in parallel | `async_execution=True` | a thread pool |
| a specialist whose model call fails | the crew waits forever; `run`'s deadline ends it | `failures.attempt`: a failure form, reported at once |
| passing work on | `context=[tasks]` | the forms, as JSON, in the lead's request |
| checking the output | `guardrail=`, retried with feedback | `tallyfold.check`, in a loop |
| a manager deciding who works | `Process.hierarchical` + `manager_llm` | chapter 6's `supervisor` |

**What CrewAI does that you do not see**, from section 5:
- **The prompt is CrewAI's**, wrapped around yours: *"This is VERY important to you, use the tools available and give your best Final Answer, your job depends on it!"*
- **It describes the form to the model in its own shorthand** — `"status": Literal` — dropping the allowed values and every field description. The model knows the four statuses only because the task's own instructions repeat them. Chapter 3's `Tool` sends the JSON schema, enum and descriptions included. The same happens to tool arguments: `{'name': {'description': None, ...}}`.
- **An agent that answers a form task in prose can end the whole crew** with `ConverterError` — shown with a stand-in LLM, on the conversion path for models without function calling.
- **Telemetry is on by default**, and the first run on a machine stops for a 20-second interactive prompt. `OTEL_SDK_DISABLED`, the usual advice, would also have switched off Langfuse in notebook 8.

**The version, again.** This is CrewAI 0.193.2, chosen because every later release pins an older `mcp` and `chromadb` than this repo runs. Newer CrewAI releases may handle tool calling differently; this notebook is a record of this version, and the dependency pin that kept it here is part of what adopting CrewAI costs.